# Sparse Matrix Formats

## Sparse Storage Model

A sparse matrix stores mostly nonzero structure. For a matrix with $m$ rows, $n$ columns, and $\mathrm{nnz}$ nonzeros, dense storage costs $mn$ values while sparse storage costs roughly $\mathrm{nnz}$ values plus index metadata.

The most common format for computation is compressed sparse row (CSR):

- `data`: nonzero values
- `indices`: column index for each nonzero
- `indptr`: offsets into `data` where each row begins and ends

In [1]:
import numpy as np
from scipy import sparse

np.set_printoptions(linewidth=100)

## A First Sparse Matrix

A standard first sparse matrix is the one-dimensional Poisson operator. It is tridiagonal, so the number of nonzeros grows linearly with the problem size.

In [2]:
def poisson_1d(n):
    diagonals = [
        -np.ones(n - 1),
        2 * np.ones(n),
        -np.ones(n - 1),
    ]
    return sparse.diags(diagonals, offsets=[-1, 0, 1], format="csr")

A = poisson_1d(8)
print(A.toarray())
print("shape:", A.shape)
print("nnz:", A.nnz)
print("density:", A.nnz / (A.shape[0] * A.shape[1]))

[[ 2. -1.  0.  0.  0.  0.  0.  0.]
 [-1.  2. -1.  0.  0.  0.  0.  0.]
 [ 0. -1.  2. -1.  0.  0.  0.  0.]
 [ 0.  0. -1.  2. -1.  0.  0.  0.]
 [ 0.  0.  0. -1.  2. -1.  0.  0.]
 [ 0.  0.  0.  0. -1.  2. -1.  0.]
 [ 0.  0.  0.  0.  0. -1.  2. -1.]
 [ 0.  0.  0.  0.  0.  0. -1.  2.]]
shape: (8, 8)
nnz: 22
density: 0.34375


## CSR Internals

CSR stores the same matrix with three arrays. The row pointer array has length `n_rows + 1`, and row `i` occupies `data[indptr[i]:indptr[i+1]]`.

In [3]:
print("data   =", A.data)
print("indices=", A.indices)
print("indptr =", A.indptr)

row = 3
lo, hi = A.indptr[row], A.indptr[row + 1]
print("row", row, "columns", A.indices[lo:hi], "values", A.data[lo:hi])

data   = [ 2. -1. -1.  2. -1. -1.  2. -1. -1.  2. -1. -1.  2. -1. -1.  2. -1. -1.  2. -1. -1.  2.]
indices= [0 1 0 1 2 1 2 3 2 3 4 3 4 5 4 5 6 5 6 7 6 7]
indptr = [ 0  2  5  8 11 14 17 20 22]
row 3 columns [2 3 4] values [-1.  2. -1.]


## Storage Scaling

For large problems, the difference between dense and sparse storage is decisive. The following estimate uses a million-by-million tridiagonal matrix. Dense storage would be impractical; CSR storage is modest.

In [4]:
n = 1_000_000
nnz = 3 * n - 2
dense_bytes = n * n * 8
csr_bytes = nnz * 8 + nnz * 4 + (n + 1) * 4

print(f"dense storage: {dense_bytes / 1e12:.1f} TB")
print(f"CSR storage:   {csr_bytes / 1e6:.1f} MB")

dense storage: 8.0 TB
CSR storage:   40.0 MB
